# DANN v1.0

`baseline v2.1` 기반 multi-view DANN notebook입니다.

- source domain: `train`
- target domain: `dev`
- `train` batch: `class loss + domain loss`
- `dev` batch: `class loss = 0`, `domain loss`만 사용
- `dev` 라벨은 모니터링/early stopping 용도로만 사용하고, class loss backprop에는 사용하지 않습니다.

In [1]:
import os
import sys
import json
import shutil
from dataclasses import dataclass
from itertools import cycle
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.autograd import Function
from torch.utils.data import Dataset, DataLoader
import timm

SRC_DIR = (Path.cwd() / '../../src').resolve()
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from augmentations import build_default_transforms
from output_paths import allocate_output_paths
from reproducibility import make_generator, seed_everything, seed_worker
from models import EMAConfig, ModelEMA

DATA_DIR = (Path.cwd() / '../../data').resolve()
assert DATA_DIR.exists(), f'data dir not found: {DATA_DIR}'
print('DATA_DIR:', DATA_DIR)

CFG = {
    'IMG_SIZE': 320,
    'EPOCHS': 30,
    'LEARNING_RATE': 3e-4,
    'BATCH_SIZE': 8,
    'SEED': 42,
    'NUM_WORKERS': 8,
    'WEIGHT_DECAY': 1e-4,
    'MIN_LR': 1e-6,
    'EMA_DECAY': 0.999,
    'EMA_USE_FOR_EVAL': True,
    'EARLY_STOPPING_PATIENCE': 8,
    'DOMAIN_LOSS_WEIGHT': 0.5,
    'GRL_WARMUP_EPOCHS': 5,
    'GRL_MAX_LAMBDA': 0.2,
    'GRL_GAMMA': 4.0,
    'BACKBONE_NAME': 'efficientnetv2_rw_s',
    'ATTN_DIM': 256,
    'NUM_HEADS': 8,
    'NUM_LAYERS': 2,
    'POS_GRID': 7,
    'DROPOUT': 0.1,
    'CLASSIFIER_HIDDEN_DIM': 512,
    'CLASSIFIER_MID_DIM': 128,
    'CLASSIFIER_DROPOUT': 0.3,
    'DOMAIN_HIDDEN_DIM': 256,
    'DOMAIN_DROPOUT': 0.2,
    'VIDEO_AUG_ENABLE': True,
    'VIDEO_AUG_CACHE': True,
    'UNSTABLE_START_MIN_SEC': 0.5,
    'UNSTABLE_START_MAX_SEC': 1.0,
    'UNSTABLE_FRAMES_MIN': 2,
    'UNSTABLE_FRAMES_MAX': 3,
    'STABLE_END_MIN_SEC': 9.0,
    'STABLE_END_MAX_SEC': 10.0,
    'STABLE_FRAMES_PER_VIDEO': 2,
}

seed_everything(CFG['SEED'])
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)
print('backbone:', CFG['BACKBONE_NAME'])


DATA_DIR: /media/hdd0/whyz/structure-stability/data
device: cuda
backbone: efficientnetv2_rw_s


In [2]:
train_df = pd.read_csv(DATA_DIR / 'train.csv', encoding='utf-8-sig')
dev_df = pd.read_csv(DATA_DIR / 'dev.csv', encoding='utf-8-sig')
test_df = pd.read_csv(DATA_DIR / 'sample_submission.csv', encoding='utf-8-sig')

print('train/dev/test:', train_df.shape, dev_df.shape, test_df.shape)
display(train_df.head())
display(dev_df.head())


train/dev/test: (1000, 2) (100, 2) (1000, 3)


,id,label
0,TRAIN_0001,unstable
1,TRAIN_0002,unstable
2,TRAIN_0003,unstable
3,TRAIN_0004,unstable
4,TRAIN_0005,stable


,id,label
0,DEV_001,unstable
1,DEV_002,unstable
2,DEV_003,unstable
3,DEV_004,stable
4,DEV_005,stable


In [3]:
class MultiViewDataset(Dataset):
    def __init__(self, df, root_dir, transform=None, is_test=False):
        self.df = df.reset_index(drop=True)
        self.root_dir = root_dir
        self.transform = transform
        self.is_test = is_test
        self.label_map = {'stable': 0, 'unstable': 1}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        sample_id = str(row['id'])

        base_dir = self.root_dir
        if ('sample_dir' in self.df.columns) and pd.notna(row.get('sample_dir', np.nan)):
            base_dir = str(row['sample_dir'])

        folder_path = os.path.join(base_dir, sample_id)
        views = []
        for name in ['front', 'top']:
            img_path = os.path.join(folder_path, f'{name}.png')
            image = Image.open(img_path).convert('RGB')
            if self.transform:
                image = self.transform(image)
            views.append(image)

        if self.is_test:
            return views

        label = self.label_map[row['label']]
        return views, label


def _extract_frame_by_sec(cap, sec, fps, frame_count):
    frame_idx = int(max(0, min(frame_count - 1, round(sec * fps))))
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    ok, frame = cap.read()
    if not ok:
        return None
    return cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)


def _extract_last_frame(cap, frame_count):
    last_idx = max(0, frame_count - 1)
    cap.set(cv2.CAP_PROP_POS_FRAMES, last_idx)
    ok, frame = cap.read()
    if not ok:
        return None
    return cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)


def _video_aug_cache_signature(cfg):
    keys = [
        'SEED',
        'UNSTABLE_START_MIN_SEC',
        'UNSTABLE_START_MAX_SEC',
        'UNSTABLE_FRAMES_MIN',
        'UNSTABLE_FRAMES_MAX',
        'STABLE_END_MIN_SEC',
        'STABLE_END_MAX_SEC',
        'STABLE_FRAMES_PER_VIDEO',
    ]
    return {k: cfg.get(k) for k in keys}


def build_video_augmented_df(train_df, data_dir, cfg):
    train_root = data_dir / 'train'
    aug_root = data_dir / 'train_video_aug'
    aug_root.mkdir(parents=True, exist_ok=True)

    cache_csv = aug_root / 'aug_df.csv'
    cache_meta = aug_root / 'cache_meta.json'
    cache_sig = _video_aug_cache_signature(cfg)

    if cfg.get('VIDEO_AUG_CACHE', True) and cache_csv.exists() and cache_meta.exists():
        try:
            meta = json.loads(cache_meta.read_text())
            if meta.get('signature') == cache_sig:
                cached_df = pd.read_csv(cache_csv)
                if not cached_df.empty:
                    cached_df['sample_dir'] = str(aug_root)
                    print(f'video aug cache hit: {len(cached_df)} samples')
                    return cached_df
        except Exception as exc:
            print('video aug cache read failed:', exc)

    for p in aug_root.glob('AUGV_*'):
        if p.is_dir():
            shutil.rmtree(p, ignore_errors=True)

    rng = np.random.default_rng(cfg['SEED'])
    saved_idx = 0
    aug_rows = []

    def save_aug(img, label):
        nonlocal saved_idx
        aug_id = f'AUGV_{saved_idx:07d}'
        out_dir = aug_root / aug_id
        out_dir.mkdir(parents=True, exist_ok=True)
        Image.fromarray(img).save(out_dir / 'front.png')
        Image.fromarray(img).save(out_dir / 'top.png')
        aug_rows.append({'id': aug_id, 'label': label, 'sample_dir': str(aug_root)})
        saved_idx += 1

    for row in tqdm(train_df.itertuples(index=False), total=len(train_df), desc='video aug', dynamic_ncols=True, ascii=True):
        sample_id = row.id
        label = row.label
        video_path = train_root / sample_id / 'simulation.mp4'
        if not video_path.exists():
            continue

        cap = cv2.VideoCapture(str(video_path))
        if not cap.isOpened():
            continue

        fps = cap.get(cv2.CAP_PROP_FPS)
        frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if fps <= 0 or frame_count <= 0:
            cap.release()
            continue

        if label == 'unstable':
            n_frames = int(rng.integers(cfg['UNSTABLE_FRAMES_MIN'], cfg['UNSTABLE_FRAMES_MAX'] + 1))
            secs = rng.uniform(cfg['UNSTABLE_START_MIN_SEC'], cfg['UNSTABLE_START_MAX_SEC'], size=n_frames)
            for sec in secs:
                frame = _extract_frame_by_sec(cap, float(sec), fps, frame_count)
                if frame is not None:
                    save_aug(frame, label)
        else:
            n_frames = int(cfg['STABLE_FRAMES_PER_VIDEO'])
            secs = rng.uniform(cfg['STABLE_END_MIN_SEC'], cfg['STABLE_END_MAX_SEC'], size=n_frames)
            for sec in secs:
                frame = _extract_frame_by_sec(cap, float(sec), fps, frame_count)
                if frame is not None:
                    save_aug(frame, label)

        cap.release()

    aug_df = pd.DataFrame(aug_rows)
    aug_df.to_csv(cache_csv, index=False)
    cache_meta.write_text(json.dumps({'signature': cache_sig}, indent=2))
    print('video aug saved:', len(aug_df))
    return aug_df


In [4]:
train_transform, test_transform = build_default_transforms(CFG['IMG_SIZE'])

train_df_for_train = train_df.copy()
train_df_for_train['sample_dir'] = str(DATA_DIR / 'train')

if CFG['VIDEO_AUG_ENABLE']:
    aug_df = build_video_augmented_df(train_df, DATA_DIR, CFG)
    if not aug_df.empty:
        train_df_for_train = pd.concat([train_df_for_train, aug_df], ignore_index=True)

train_domain_df = train_df_for_train.copy()
dev_domain_df = dev_df.copy()
dev_domain_df['sample_dir'] = str(DATA_DIR / 'dev')
dev_eval_df = dev_df.copy()
dev_eval_df['sample_dir'] = str(DATA_DIR / 'dev')
test_infer_df = test_df.copy()
test_infer_df['sample_dir'] = str(DATA_DIR / 'test')

train_dataset = MultiViewDataset(train_domain_df, str(DATA_DIR / 'train'), train_transform, is_test=False)
dev_domain_dataset = MultiViewDataset(dev_domain_df, str(DATA_DIR / 'dev'), train_transform, is_test=False)
dev_eval_dataset = MultiViewDataset(dev_eval_df, str(DATA_DIR / 'dev'), test_transform, is_test=False)
test_dataset = MultiViewDataset(test_infer_df, str(DATA_DIR / 'test'), test_transform, is_test=True)

loader_kwargs = dict(
    batch_size=CFG['BATCH_SIZE'],
    num_workers=CFG['NUM_WORKERS'],
    pin_memory=(device.type == 'cuda'),
)

g = make_generator(CFG['SEED'])
train_loader = DataLoader(train_dataset, shuffle=True, worker_init_fn=seed_worker, generator=g, **loader_kwargs)
dev_domain_loader = DataLoader(dev_domain_dataset, shuffle=True, worker_init_fn=seed_worker, generator=make_generator(CFG['SEED'] + 1), **loader_kwargs)
dev_eval_loader = DataLoader(dev_eval_dataset, shuffle=False, **loader_kwargs)
test_loader = DataLoader(test_dataset, shuffle=False, **loader_kwargs)

print('train_dataset:', len(train_dataset))
print('dev_domain_dataset:', len(dev_domain_dataset))
print('dev_eval_dataset:', len(dev_eval_dataset))
print('test_dataset:', len(test_dataset))


video aug cache hit: 2000 samples
train_dataset: 3000
dev_domain_dataset: 100
dev_eval_dataset: 100
test_dataset: 1000


In [ ]:
@dataclass(frozen=True)
class DANNConfig:
    backbone_name: str = CFG['BACKBONE_NAME']
    pretrained: bool = True
    attn_dim: int = CFG['ATTN_DIM']
    num_heads: int = CFG['NUM_HEADS']
    num_layers: int = CFG['NUM_LAYERS']
    pos_grid: int = CFG['POS_GRID']
    dropout: float = CFG['DROPOUT']
    classifier_hidden_dim: int = CFG['CLASSIFIER_HIDDEN_DIM']
    classifier_mid_dim: int = CFG['CLASSIFIER_MID_DIM']
    classifier_dropout: float = CFG['CLASSIFIER_DROPOUT']
    domain_hidden_dim: int = CFG['DOMAIN_HIDDEN_DIM']
    domain_dropout: float = CFG['DOMAIN_DROPOUT']
    out_dim: int = 1


class GradientReversalFunction(Function):
    @staticmethod
    def forward(ctx, x, lambda_):
        ctx.lambda_ = lambda_
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.lambda_ * grad_output, None


def grad_reverse(x, lambda_=1.0):
    return GradientReversalFunction.apply(x, lambda_)


class CrossAttentionBlock(nn.Module):
    def __init__(self, dim: int, num_heads: int, dropout: float = 0.1):
        super().__init__()
        self.norm_q = nn.LayerNorm(dim)
        self.norm_kv = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, num_heads, dropout=dropout, batch_first=True)

    def forward(self, q_tokens, kv_tokens):
        q = self.norm_q(q_tokens)
        kv = self.norm_kv(kv_tokens)
        attn_out, _ = self.attn(q, kv, kv, need_weights=False)
        return q_tokens + attn_out


class MultiViewDANN(nn.Module):
    def __init__(self, config: DANNConfig | None = None):
        super().__init__()
        self.config = config or DANNConfig()
        self.backbone = timm.create_model(
            self.config.backbone_name,
            pretrained=self.config.pretrained,
            num_classes=0,
            global_pool='',
        )
        feature_dim = self.backbone.num_features
        self.proj = nn.Conv2d(feature_dim, self.config.attn_dim, kernel_size=1, bias=False)
        self.pos_embed = nn.Parameter(torch.randn(1, self.config.attn_dim, self.config.pos_grid, self.config.pos_grid) * 0.02)
        self.cross_12 = nn.ModuleList([CrossAttentionBlock(self.config.attn_dim, self.config.num_heads, self.config.dropout) for _ in range(self.config.num_layers)])
        self.cross_21 = nn.ModuleList([CrossAttentionBlock(self.config.attn_dim, self.config.num_heads, self.config.dropout) for _ in range(self.config.num_layers)])
        self.classifier = nn.Sequential(
            nn.Linear(self.config.attn_dim * 2, self.config.classifier_hidden_dim),
            nn.BatchNorm1d(self.config.classifier_hidden_dim),
            nn.ReLU(),
            nn.Dropout(self.config.classifier_dropout),
            nn.Linear(self.config.classifier_hidden_dim, self.config.classifier_mid_dim),
            nn.ReLU(),
            nn.Linear(self.config.classifier_mid_dim, self.config.out_dim),
        )
        self.domain_classifier = nn.Sequential(
            nn.Linear(self.config.attn_dim * 2, self.config.domain_hidden_dim),
            nn.ReLU(),
            nn.Dropout(self.config.domain_dropout),
            nn.Linear(self.config.domain_hidden_dim, 1),
        )

    def _to_tokens(self, feat_map):
        feat_map = self.proj(feat_map)
        pos = F.interpolate(self.pos_embed, size=feat_map.shape[-2:], mode='bilinear', align_corners=False)
        feat_map = feat_map + pos
        return feat_map.flatten(2).transpose(1, 2)

    def extract_features(self, views):
        f1 = self.backbone.forward_features(views[0])
        f2 = self.backbone.forward_features(views[1])
        t1 = self._to_tokens(f1)
        t2 = self._to_tokens(f2)
        for blk12, blk21 in zip(self.cross_12, self.cross_21):
            t1_prev, t2_prev = t1, t2
            t1 = blk12(t1_prev, t2_prev)
            t2 = blk21(t2_prev, t1_prev)
        return torch.cat([t1.mean(dim=1), t2.mean(dim=1)], dim=1)

    def forward_class_logits(self, views):
        feat = self.extract_features(views)
        return self.classifier(feat)

    def forward_domain_logits(self, views, lambda_=1.0):
        feat = self.extract_features(views)
        feat = grad_reverse(feat, lambda_)
        return self.domain_classifier(feat)


In [ ]:
def compute_lambda(epoch_idx, step_idx, steps_per_epoch, total_epochs, warmup_epochs=0, max_lambda=1.0, gamma=10.0):
    total_steps = max(1, total_epochs * steps_per_epoch)
    current_step = max(0, (epoch_idx - 1) * steps_per_epoch + step_idx)
    progress = current_step / total_steps
    warmup_progress = warmup_epochs / max(1, total_epochs)
    if progress <= warmup_progress:
        return 0.0
    effective_progress = (progress - warmup_progress) / max(1e-8, 1.0 - warmup_progress)
    lambda_base = 2.0 / (1.0 + np.exp(-gamma * effective_progress)) - 1.0
    return float(max_lambda * lambda_base)


def cat_views(views_a, views_b, device):
    out = []
    for va, vb in zip(views_a, views_b):
        out.append(torch.cat([va.to(device), vb.to(device)], dim=0))
    return out


def train_one_epoch_dann(model, train_loader, dev_loader, class_criterion, domain_criterion, optimizer, device, epoch_idx, total_epochs, domain_loss_weight=0.5, ema=None):
    model.train()
    dev_iter = cycle(dev_loader)

    total_loss_sum = 0.0
    class_loss_sum = 0.0
    domain_loss_sum = 0.0
    domain_acc_sum = 0.0

    steps = len(train_loader)
    for step_idx, (train_views, train_labels) in enumerate(tqdm(train_loader, desc='Training DANN', dynamic_ncols=True, ascii=True), start=1):
        dev_views, _ = next(dev_iter)
        train_views = [v.to(device) for v in train_views]
        dev_views = [v.to(device) for v in dev_views]
        train_labels = train_labels.to(device).float()

        optimizer.zero_grad()

        class_logits = model.forward_class_logits(train_views).view(-1)
        class_loss = class_criterion(class_logits, train_labels)

        domain_views = [torch.cat([tv, dv], dim=0) for tv, dv in zip(train_views, dev_views)]
        domain_labels = torch.cat([
            torch.zeros(train_views[0].size(0), device=device),
            torch.ones(dev_views[0].size(0), device=device),
        ], dim=0)
        grl_lambda = compute_lambda(
            epoch_idx,
            step_idx - 1,
            steps,
            total_epochs,
            warmup_epochs=CFG['GRL_WARMUP_EPOCHS'],
            max_lambda=CFG['GRL_MAX_LAMBDA'],
            gamma=CFG['GRL_GAMMA'],
        )
        domain_logits = model.forward_domain_logits(domain_views, lambda_=grl_lambda).view(-1)
        domain_loss = domain_criterion(domain_logits, domain_labels)

        total_loss = class_loss + domain_loss_weight * domain_loss
        total_loss.backward()
        optimizer.step()
        if ema is not None:
            ema.update(model)

        with torch.no_grad():
            domain_probs = torch.sigmoid(domain_logits)
            domain_acc = ((domain_probs > 0.5) == domain_labels).float().mean().item()

        total_loss_sum += float(total_loss.item())
        class_loss_sum += float(class_loss.item())
        domain_loss_sum += float(domain_loss.item())
        domain_acc_sum += float(domain_acc)

    n = max(1, len(train_loader))
    return {
        'train_total_loss': total_loss_sum / n,
        'train_class_loss': class_loss_sum / n,
        'train_domain_loss': domain_loss_sum / n,
        'train_domain_acc': domain_acc_sum / n,
    }


def evaluate_classification(model, loader, device):
    model.eval()
    all_probs = []
    all_labels = []
    with torch.no_grad():
        for views, labels in tqdm(loader, desc='Dev Eval', dynamic_ncols=True, ascii=True):
            views = [v.to(device) for v in views]
            labels = labels.to(device).float()
            logits = model.forward_class_logits(views).view(-1)
            probs = torch.sigmoid(logits)
            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    all_probs = np.array(all_probs, dtype=np.float64)
    all_labels = np.array(all_labels, dtype=np.float64)
    p = np.clip(all_probs, 1e-15, 1 - 1e-15)
    logloss = float(-np.mean(all_labels * np.log(p) + (1.0 - all_labels) * np.log(1.0 - p)))
    acc = float(np.mean((all_probs > 0.5) == all_labels))
    return logloss, acc, all_probs, all_labels


def predict_test_probs(model, loader, device):
    model.eval()
    all_probs = []
    with torch.no_grad():
        for views in tqdm(loader, desc='Test Inference', dynamic_ncols=True, ascii=True):
            views = [v.to(device) for v in views]
            logits = model.forward_class_logits(views).view(-1)
            probs = torch.sigmoid(logits)
            all_probs.extend(probs.cpu().numpy())
    return np.array(all_probs, dtype=np.float64)


In [7]:
model = MultiViewDANN(DANNConfig()).to(device)
class_criterion = nn.BCEWithLogitsLoss()
domain_criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=CFG['LEARNING_RATE'], weight_decay=CFG['WEIGHT_DECAY'])
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG['EPOCHS'], eta_min=CFG['MIN_LR'])
ema = ModelEMA(model, EMAConfig(decay=CFG['EMA_DECAY']))

artifacts = allocate_output_paths(experiment_name='dann', major_version='v1.0')
best_model_path = artifacts['weight_path']
submission_path = artifacts['submission_path']
history_path = (Path.cwd() / '../../outputs/eda_preprocessing/dann_v1.0_history.csv').resolve()
history_path.parent.mkdir(parents=True, exist_ok=True)
print('Artifact version:', artifacts['version'])
print('best_model_path:', best_model_path)
print('submission_path:', submission_path)

best_logloss = float('inf')
best_epoch = -1
patience_left = CFG['EARLY_STOPPING_PATIENCE']
history = []

for epoch in range(1, CFG['EPOCHS'] + 1):
    train_metrics = train_one_epoch_dann(
        model,
        train_loader,
        dev_domain_loader,
        class_criterion,
        domain_criterion,
        optimizer,
        device,
        epoch_idx=epoch,
        total_epochs=CFG['EPOCHS'],
        domain_loss_weight=CFG['DOMAIN_LOSS_WEIGHT'],
        ema=ema,
    )

    eval_model = ema.ema_model if CFG['EMA_USE_FOR_EVAL'] else model
    dev_logloss, dev_acc, _, _ = evaluate_classification(eval_model, dev_eval_loader, device)

    improved = dev_logloss < best_logloss
    if improved:
        best_logloss = dev_logloss
        best_epoch = epoch
        patience_left = CFG['EARLY_STOPPING_PATIENCE']
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'ema_state_dict': ema.ema_model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'cfg': CFG,
            'dev_logloss': dev_logloss,
            'dev_acc': dev_acc,
        }, best_model_path)
    else:
        patience_left -= 1

    scheduler.step()
    row = {
        'epoch': epoch,
        **train_metrics,
        'dev_logloss': float(dev_logloss),
        'dev_acc': float(dev_acc),
        'lr': float(optimizer.param_groups[0]['lr']),
        'improved': bool(improved),
        'patience_left': int(patience_left),
    }
    history.append(row)
    print(row)

    if patience_left < 0:
        print('early stop:', epoch)
        break



Artifact version: v1.0.1
best_model_path: /media/hdd0/whyz/structure-stability/outputs/weights/dann_v1.0.1.pt
submission_path: /media/hdd0/whyz/structure-stability/outputs/submissions/dann_v1.0.1.csv


Dev Eval: 100%|##########| 13/13 [00:00<00:00, 16.01it/s]


{'epoch': 1, 'train_total_loss': 0.5362689473628998, 'train_class_loss': 0.3325702532132467, 'train_domain_loss': 0.4073973876039187, 'train_domain_acc': 0.8350000015894572, 'dev_logloss': 0.6732682466346758, 'dev_acc': 0.52, 'lr': 0.0002991810233575568, 'improved': True, 'patience_left': 8}


Dev Eval: 100%|##########| 13/13 [00:00<00:00, 16.39it/s]


{'epoch': 2, 'train_total_loss': 0.3577403035859267, 'train_class_loss': 0.19508023004730543, 'train_domain_loss': 0.3253201474348704, 'train_domain_acc': 0.8610000012715657, 'dev_logloss': 0.58078341547847, 'dev_acc': 0.86, 'lr': 0.0002967330663097039, 'improved': True, 'patience_left': 8}


...

{'epoch': 11, 'train_total_loss': 0.43107569630940756, 'train_class_loss': 0.08705798382870852, 'train_domain_loss': 0.6880354235967, 'train_domain_acc': 0.6124444457689922, 'dev_logloss': 0.2984064411285478, 'dev_acc': 0.82, 'lr': 0.0002113071281398321, 'improved': False, 'patience_left': 6}



Dev Eval: 100%|##########| 13/13 [00:00<00:00, 16.13it/s]

{'epoch': 12, 'train_total_loss': 0.4240052354335785, 'train_class_loss': 0.09483391892661651, 'train_domain_loss': 0.6583426347573598, 'train_domain_acc': 0.6094444454908371, 'dev_logloss': 0.30418955200770414, 'dev_acc': 0.81, 'lr': 0.00019669804065905458, 'improved': False, 'patience_left': 5}



Training DANN:  54%|#####4    | 203/375 [01:26<01:13,  2.35it/s]


KeyboardInterrupt: 

In [8]:

history_df = pd.DataFrame(history)
history_df.to_csv(history_path, index=False)
print('saved history:', history_path)

checkpoint = torch.load(best_model_path, map_location=device, weights_only=False)
best_state = checkpoint['ema_state_dict'] if CFG['EMA_USE_FOR_EVAL'] else checkpoint['model_state_dict']
model.load_state_dict(best_state)

final_dev_logloss, final_dev_acc, dev_probs, dev_labels = evaluate_classification(model, dev_eval_loader, device)
print('best_epoch:', best_epoch)
print('final_dev_logloss:', final_dev_logloss)
print('final_dev_acc:', final_dev_acc)

saved history: /media/hdd0/whyz/structure-stability/outputs/eda_preprocessing/dann_v1.0_history.csv


Dev Eval: 100%|##########| 13/13 [00:00<00:00, 15.86it/s]

best_epoch: 9
final_dev_logloss: 0.12175931912737932
final_dev_acc: 0.98


In [ ]:
def _center_crop_and_resize(x, crop_ratio=0.95):
    b, c, h, w = x.shape
    ch, cw = int(h * crop_ratio), int(w * crop_ratio)
    y1 = (h - ch) // 2
    x1 = (w - cw) // 2
    cropped = x[:, :, y1:y1 + ch, x1:x1 + cw]
    return F.interpolate(cropped, size=(h, w), mode='bilinear', align_corners=False)


def apply_tta_to_views(views, tta_name):
    if tta_name == 'none':
        return views
    if tta_name == 'hflip':
        return [torch.flip(v, dims=[3]) for v in views]
    if tta_name == 'crop95':
        return [_center_crop_and_resize(v, crop_ratio=0.95) for v in views]
    raise ValueError(f'Unknown TTA: {tta_name}')


def predict_probs_with_tta_dann(model, loader, device, tta_names=None, has_labels=False, desc='Inference TTA'):
    if tta_names is None:
        tta_names = ['none']

    model.eval()
    all_probs = []
    all_labels = []

    with torch.no_grad():
        for batch in tqdm(loader, desc=desc, dynamic_ncols=True, ascii=True):
            if has_labels:
                views, labels = batch
                labels = labels.to(device).float()
                all_labels.extend(labels.cpu().numpy())
            else:
                views = batch

            views = [v.to(device) for v in views]

            probs_sum = None
            for tta_name in tta_names:
                tta_views = apply_tta_to_views(views, tta_name)
                logits = model.forward_class_logits(tta_views).view(-1)
                probs = torch.sigmoid(logits)
                probs_sum = probs if probs_sum is None else (probs_sum + probs)

            probs_avg = probs_sum / len(tta_names)
            all_probs.extend(probs_avg.cpu().numpy())

    all_probs = np.array(all_probs, dtype=np.float64)
    if has_labels:
        return all_probs, np.array(all_labels, dtype=np.float64)
    return all_probs


def evaluate_tta_on_dev_dann(model, loader, device, tta_candidates):
    rows = []
    for tta_names in tta_candidates:
        probs, labels = predict_probs_with_tta_dann(
            model, loader, device, tta_names=tta_names, has_labels=True,
            desc=f'Dev TTA {tta_names}'
        )
        p = np.clip(probs, 1e-15, 1 - 1e-15)
        logloss_score = -np.mean(labels * np.log(p) + (1.0 - labels) * np.log(1.0 - p))
        acc_score = np.mean((probs > 0.5) == labels)
        rows.append({
            'tta_names': tta_names,
            'n_tta': len(tta_names),
            'dev_logloss': float(logloss_score),
            'dev_acc': float(acc_score),
        })
    return pd.DataFrame(rows).sort_values('dev_logloss', ascending=True).reset_index(drop=True)


tta_candidates = CFG.get('TTA_CANDIDATES', [
    ['none'],
    ['none', 'hflip'],
    ['none', 'hflip', 'crop95'],
])

tta_result_df = evaluate_tta_on_dev_dann(model, dev_eval_loader, device, tta_candidates)
display(tta_result_df)

best_tta_names = tta_result_df.iloc[0]['tta_names']
tta_dev_logloss = float(tta_result_df.iloc[0]['dev_logloss'])
tta_dev_acc = float(tta_result_df.iloc[0]['dev_acc'])
print('best_tta_names:', best_tta_names)
print('tta_dev_logloss:', tta_dev_logloss)
print('tta_dev_acc:', tta_dev_acc)

test_probs = predict_probs_with_tta_dann(
    model,
    test_loader,
    device,
    tta_names=best_tta_names,
    has_labels=False,
    desc='Test Inference TTA',
)
sub = test_df.copy()
sub['unstable_prob'] = test_probs
sub['stable_prob'] = 1.0 - test_probs
sub.to_csv(submission_path, index=False, encoding='utf-8-sig')
print('saved submission:', submission_path)

summary = pd.DataFrame([
    {
        'experiment': 'DANN_v1.0',
        'artifact_version': artifacts['version'],
        'backbone': CFG['BACKBONE_NAME'],
        'best_epoch': best_epoch,
        'dev_logloss': final_dev_logloss,
        'dev_acc': final_dev_acc,
        'tta_names': str(best_tta_names),
        'tta_dev_logloss': tta_dev_logloss,
        'tta_dev_acc': tta_dev_acc,
        'domain_loss_weight': CFG['DOMAIN_LOSS_WEIGHT'],
        'grl_warmup_epochs': CFG['GRL_WARMUP_EPOCHS'],
        'grl_max_lambda': CFG['GRL_MAX_LAMBDA'],
        'grl_gamma': CFG['GRL_GAMMA'],
    }
])
summary_path = (Path.cwd() / '../../outputs/eda_preprocessing/dann_v1.0_summary.csv').resolve()
summary.to_csv(summary_path, index=False)
print('saved summary:', summary_path)
summary


Dev TTA ['none', 'hflip', 'crop95']: 100%|##########| 13/13 [00:01<00:00,  7.48it/s]


,tta_names,n_tta,dev_logloss,dev_acc
0,[none],1,0.121759,0.98
1,"[none, hflip, crop95]",3,0.121978,0.98
2,"[none, hflip]",2,0.129198,0.96


best_tta_names: ['none']
tta_dev_logloss: 0.12175931912737932
tta_dev_acc: 0.98


Test Inference TTA: 100%|##########| 125/125 [00:05<00:00, 24.58it/s]

saved submission: /media/hdd0/whyz/structure-stability/outputs/submissions/dann_v1.0.1.csv
saved summary: /media/hdd0/whyz/structure-stability/outputs/eda_preprocessing/dann_v1.0_summary.csv


,experiment,artifact_version,backbone,best_epoch,dev_logloss,dev_acc,tta_names,tta_dev_logloss,tta_dev_acc,domain_loss_weight,grl_warmup_epochs,grl_max_lambda,grl_gamma
0,DANN_v1.0,v1.0.1,efficientnetv2_rw_s,9,0.121759,0.98,['none'],0.121759,0.98,0.5,5,0.2,4.0


: 